In [21]:
import math
import numpy as np, pandas as pd, scipy.sparse as sp
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors
from tqdm.auto import tqdm

SEED = 42
BOBOT = {"view": 1, "addtocart": 3, "transaction": 5}   

p1 = pd.read_csv("data/p1/events.csv")
p2 = pd.read_csv("data/p2/events.csv")

p1["w"] = p1["event"].map(BOBOT)
pair = (p1.groupby(["visitorid", "itemid"], as_index=False)["w"].max()
          .rename(columns={"visitorid": "u", "itemid": "i"}))

user_ids = np.sort(pair["u"].unique())
item_ids = np.sort(pair["i"].unique())
u_idx = pd.Index(user_ids).get_indexer(pair["u"])
i_idx = pd.Index(item_ids).get_indexer(pair["i"])

X = sp.csr_matrix((pair["w"].astype(float), (u_idx, i_idx)),
                  shape=(len(user_ids), len(item_ids)))
print("Matriks User x Item:", X.shape)

Matriks User x Item: (650722, 168452)


## Subbab 2.2 nomor 1a

In [22]:
REL_EVENTS = ["addtocart", "transaction"]       

seen_items = pair.groupby("u")["i"].apply(set).to_dict()                     
test_items = (p2[p2["event"].isin(REL_EVENTS)]
              .groupby("visitorid")["itemid"].apply(set).to_dict())            

kandidat_user = sorted(u for u in test_items
                       if u in seen_items and len(seen_items[u]) >= 3
                       and len(test_items[u] - seen_items[u]) > 0)
print("Jumlah kandidat user:", len(kandidat_user))
assert kandidat_user, "Tidak ada kandidat; longgarkan syarat atau ubah periode"

rng = np.random.default_rng(SEED)
TARGET = int(rng.choice(kandidat_user))
relevan = test_items[TARGET] - seen_items[TARGET]
print("User terpilih:", TARGET, "| item di P1:", len(seen_items[TARGET]), "| item relevan di P2:", sorted(relevan))

Jumlah kandidat user: 442
User terpilih: 163049 | item di P1: 19 | item relevan di P2: [215051]


In [23]:
Xn = normalize(X, norm="l2", axis=1).tocsr()    
XnT = Xn.T.tocsr()
uid_to_row = {int(u): n for n, u in enumerate(user_ids)}
n_item_user = X.getnnz(axis=1)

def items_of(j):
    return item_ids[X.indices[X.indptr[j]:X.indptr[j + 1]]].tolist()

def tetangga_mentah(user_id, k=5):
    """Cara lama: k user dengan cosine tertinggi, tanpa syarat apa pun."""
    t = uid_to_row[user_id]
    s = (Xn[t] @ XnT).tocoo()
    m = s.col != t
    cols, vals = s.col[m], s.data[m]
    order = np.lexsort((user_ids[cols], -vals))[:k]
    return [(int(user_ids[cols[o]]), float(vals[o])) for o in order]

for u, s in tetangga_mentah(TARGET, 5):
    its = set(items_of(uid_to_row[u]))
    print(f"user {u} | cosine {s:.4f} | jumlah item {len(its)} | item BARU bagi target: {len(its - seen_items[TARGET])}")

user 555036 | cosine 0.7716 | jumlah item 2 | item BARU bagi target: 0
user 90468 | cosine 0.7625 | jumlah item 1 | item BARU bagi target: 0
user 398889 | cosine 0.7625 | jumlah item 1 | item BARU bagi target: 0
user 659146 | cosine 0.7625 | jumlah item 1 | item BARU bagi target: 0
user 660531 | cosine 0.7625 | jumlah item 1 | item BARU bagi target: 0


In [24]:
pop = np.asarray(X.sum(axis=0)).ravel()                        
pop_order = item_ids[np.argsort(-pop, kind="stable")]          

def knn_tetangga(user_id, k=5, pool=1000, min_items=2):
    t = uid_to_row[user_id]
    s = (Xn[t] @ XnT).tocoo()
    m = s.col != t
    cols, vals = s.col[m], s.data[m]
    order = np.lexsort((user_ids[cols], -vals))[:pool]
    hasil = []
    for o in order:
        j = int(cols[o])
        if n_item_user[j] < min_items:
            continue
        if not (set(items_of(j)) - seen_items[user_id]):
            continue
        hasil.append((int(user_ids[j]), float(vals[o])))
        if len(hasil) == k:
            break
    return hasil

def rekomendasi_knn(user_id, k=5, n=5, isi_populer=True):
    """Kembalikan n item: (itemId, skor, sumber)."""
    skor = {}
    for u, s in knn_tetangga(user_id, k):
        j = uid_to_row[u]
        for col, w in zip(X.indices[X.indptr[j]:X.indptr[j + 1]], X.data[X.indptr[j]:X.indptr[j + 1]]):
            itm = int(item_ids[col])
            if itm in seen_items[user_id]:
                continue
            skor[itm] = skor.get(itm, 0.0) + s * w
    hasil = [(i, sc, "KNN") for i, sc in sorted(skor.items(), key=lambda kv: (-kv[1], kv[0]))[:n]]
    if isi_populer and len(hasil) < n:
        dipakai = {i for i, _, _ in hasil}
        for itm in pop_order:
            itm = int(itm)
            if itm in seen_items[user_id] or itm in dipakai:
                continue
            hasil.append((itm, 0.0, "populer"))
            if len(hasil) == n:
                break
    return hasil

K = 5
tetangga = knn_tetangga(TARGET, K)
print(f"{K} tetangga terdekat user {TARGET} (hanya yang menyumbang item baru):")
tabel_tetangga = pd.DataFrame(tetangga, columns=["userId", "cosine"])
tabel_tetangga["jumlah item"] = [len(items_of(uid_to_row[u])) for u in tabel_tetangga["userId"]]
tabel_tetangga["item baru bagi target"] = [len(set(items_of(uid_to_row[u])) - seen_items[TARGET])
                                           for u in tabel_tetangga["userId"]]
display(tabel_tetangga)

5 tetangga terdekat user 163049 (hanya yang menyumbang item baru):


,userId,cosine,jumlah item,item baru bagi target
0,939751,0.733709,3,2
1,65270,0.626916,10,1
2,1029863,0.611354,14,3
3,182312,0.609994,9,1
4,1312606,0.592137,13,3


# Subbab 2.1 nomor 1b 

In [25]:
N_IKLAN = 3
rank5 = rekomendasi_knn(TARGET, k=K, n=5)         
iklan = rank5[:N_IKLAN]                            

print(f"3 item yang diiklankan untuk user {TARGET} (k = {K}):")
display(pd.DataFrame(iklan, columns=["itemId", "skor", "sumber"]))
assert len(iklan) == N_IKLAN, "Item iklan kurang dari 3"

3 item yang diiklankan untuk user 163049 (k = 5):


,itemId,skor,sumber
0,269085,1.325846,KNN
1,9877,1.203491,KNN
2,322341,1.203491,KNN


# Subbab 2.2 nomor 2a dan 2b

In [26]:
def hit_at_k(rec, rel, k):
    return int(any(i in rel for i in rec[:k]))

def ndcg_at_k(rec, rel, k):
    dcg = sum((1.0 if i in rel else 0.0) / math.log2(pos + 2) for pos, i in enumerate(rec[:k]))
    idcg = sum(1.0 / math.log2(pos + 2) for pos in range(min(len(rel), k)))
    return dcg / idcg if idcg > 0 else 0.0

rec_ids = [i for i, _, _ in rank5]
rincian = pd.DataFrame({
    "posisi": range(1, len(rec_ids) + 1),
    "itemId": rec_ids,
    "skor": [round(s, 4) for _, s, _ in rank5],
    "sumber": [src for _, _, src in rank5],
    "relevan (rel)": [int(i in relevan) for i in rec_ids],
})
rincian["diskon 1/log2(i+1)"] = 1 / np.log2(rincian["posisi"] + 1)
rincian["kontribusi DCG"] = rincian["relevan (rel)"] * rincian["diskon 1/log2(i+1)"]
display(rincian)

print("Item relevan (ground truth):", sorted(relevan))
print("Hit@5  =", hit_at_k(rec_ids, relevan, 5))
print("NDCG@5 =", round(ndcg_at_k(rec_ids, relevan, 5), 4))
print("Pembanding hanya 3 item iklan -> Hit@3 =", hit_at_k(rec_ids, relevan, 3),
      "| NDCG@3 =", round(ndcg_at_k(rec_ids, relevan, 3), 4))

,posisi,itemId,skor,sumber,relevan (rel),diskon 1/log2(i+1),kontribusi DCG
0,1,269085,1.3258,KNN,0,1.000000,0.0
1,2,9877,1.2035,KNN,0,0.630930,0.0
2,3,322341,1.2035,KNN,0,0.500000,0.0
3,4,142324,0.7337,KNN,0,0.430677,0.0
4,5,82377,0.6269,KNN,0,0.386853,0.0


Item relevan (ground truth): [215051]
Hit@5  = 0
NDCG@5 = 0.0
Pembanding hanya 3 item iklan -> Hit@3 = 0 | NDCG@3 = 0.0


In [27]:
N_EVAL = min(200, len(kandidat_user))
rng = np.random.default_rng(SEED)
sampel = rng.choice(kandidat_user, N_EVAL, replace=False)

hits, ndcgs, pakai_populer = [], [], []
for u in tqdm(sampel):
    u = int(u)
    rs = rekomendasi_knn(u, k=K, n=5)
    rec = [i for i, _, _ in rs]
    rel = test_items[u] - seen_items[u]
    hits.append(hit_at_k(rec, rel, 5)); ndcgs.append(ndcg_at_k(rec, rel, 5))
    pakai_populer.append(any(src == "populer" for _, _, src in rs))

print(f"{N_EVAL} user | Hit@5 rata-rata = {np.mean(hits):.4f} | NDCG@5 rata-rata = {np.mean(ndcgs):.4f}")
print(f"User yang daftar rekomendasinya sebagian diisi item populer: {np.mean(pakai_populer):.1%}")

  0%|          | 0/200 [00:00<?, ?it/s]

200 user | Hit@5 rata-rata = 0.0500 | NDCG@5 rata-rata = 0.0274
User yang daftar rekomendasinya sebagian diisi item populer: 8.5%


Pemilihan user dilakukan secara acak dan reproducible dari 442 kandidat yang memenuhi kriteria, yaitu memiliki event `addtocart` atau `transaction` pada P2, memiliki minimal tiga item di P1, serta berinteraksi dengan setidaknya satu item baru di P2, menggunakan `np.random.default_rng(42).choice(...)` pada daftar kandidat yang telah diurutkan, sehingga terpilih user 163049 dengan 19 item di P1 dan item relevan 215051 di P2. Skrip KNN yang diterapkan menggunakan pendekatan user-based, yaitu matriks User × Item berbobot 1, 3, dan 5 dinormalisasi menggunakan L2, kemudian cosine similarity antara user target dan seluruh user dihitung melalui perkalian matriks sparse. Dari 1.000 user dengan kemiripan tertinggi, dipilih lima tetangga yang memiliki minimal dua item dan setidaknya satu item baru untuk direkomendasikan, guna menghindari kelemahan KNN dasar yang dapat memilih user dengan kemiripan tinggi tetapi kontribusi rekomendasinya terbatas, seperti user 555036 dengan cosine similarity 0,77 yang hanya memiliki dua item dan tidak menyumbangkan item baru. Skor rekomendasi dihitung dengan menjumlahkan hasil perkalian nilai cosine similarity dan bobot interaksi setiap item milik tetangga yang belum pernah dilihat user target, kemudian diurutkan berdasarkan skor tertinggi dengan `itemId` terkecil sebagai pemecah seri, serta dilengkapi item terpopuler apabila jumlah rekomendasi kurang dari lima. Lima tetangga terpilih adalah user 939751 (0,7337), 65270 (0,6269), 1029863 (0,6114), 182312 (0,6100), dan 1312606 (0,5921), sedangkan tiga item yang direkomendasikan untuk \\(k=5\\) adalah 269085 dengan skor 1,3258, 9877 dengan skor 1,2035, dan 322341 dengan skor 1,2035, diikuti item 142324 dan 82377 dalam daftar top-5. Namun, item relevan 215051 tidak muncul dalam rekomendasi sehingga nilai Hit\@5 dan NDCG\@5 untuk user tersebut adalah 0, sementara evaluasi terhadap 200 user acak menghasilkan rata-rata Hit\@5 sebesar 0,05 dan NDCG\@5 sebesar 0,0274. Rendahnya performa tersebut dapat dikaitkan dengan karakteristik data yang sangat sparse serta kondisi cold-start, karena hanya sekitar 9,85% user pada P2 yang telah muncul di P1, sehingga KNN berbasis user mengalami kesulitan dalam mengidentifikasi item relevan di antara ratusan ribu kandidat.